# IMPORT

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from collections import deque
import random
import math

## Bài 2 – Multiscale Hough Transform

### 2.3 Cài đặt Top-k peaks với NMS trên Accumulator

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def get_top_k_peaks_nms(accumulator, k, nhood_size=11):
    """
    Tìm k đỉnh cao nhất trên accumulator với NMS bằng NumPy 
    - nhood_size: kích thước cửa sổ NMS (phải là số lẻ)
    """
    acc_copy = np.copy(accumulator)
    peaks_rho = []
    peaks_theta = []
    
    offset = nhood_size // 2
    
    for _ in range(k):
        # 1. Tìm vị trí có giá trị vote cao nhất hiện tại
        idx = np.argmax(acc_copy)
        rho_idx, theta_idx = np.unravel_index(idx, acc_copy.shape)
        
        # Nếu đỉnh cao nhất là 0 (không còn đường thẳng nào), dừng lại
        if acc_copy[rho_idx, theta_idx] == 0:
            break
            
        peaks_rho.append(rho_idx)
        peaks_theta.append(theta_idx)
        
        # 2. Áp dụng NMS: Xóa (gán = 0) các giá trị trong cửa sổ xung quanh đỉnh vừa tìm được
        r_min = max(0, rho_idx - offset)
        r_max = min(acc_copy.shape[0], rho_idx + offset + 1)
        t_min = max(0, theta_idx - offset)
        t_max = min(acc_copy.shape[1], theta_idx + offset + 1)
        
        acc_copy[r_min:r_max, t_min:t_max] = 0
        
    return peaks_rho, peaks_theta

# (Mô phỏng chạy trên ảnh bàn cờ)
# Giả sử ta đã có ma trận accumulator từ ảnh bàn cờ
print("Thử nghiệm hàm tìm đỉnh với k=1, 3, 5...")
# top_1 = get_top_k_peaks_nms(accumulator, k=1)
# top_3 = get_top_k_peaks_nms(accumulator, k=3)
# top_5 = get_top_k_peaks_nms(accumulator, k=5)

So sánh kết quả k = 1, 3, 5 trên ảnh bàn cờ 
- k=1 hoặc k=3: Thuật toán sẽ chỉ trả về 1 hoặc 3 đường thẳng có lượng vote cao nhất (đậm nhất/dài nhất). Ảnh bàn cờ là một lưới gồm rất nhiều đường ngang và dọc đan xen. Việc chọn k quá nhỏ sẽ bỏ sót hoàn toàn cấu trúc hình học (topology) cơ bản của bàn cờ
- k=5: Bắt đầu hiển thị đủ số lượng đường để tạo thành sự giao cắt (ví dụ: 3 đường dọc, 2 đường ngang)
- Kết luận: Trong 3 giá trị trên, k=5 là phù hợp nhất vì nó cung cấp đủ số lượng cạnh để quan sát được hình dáng góc cạnh của lưới bàn cờ. Tuy nhiên, trong thực tế, giá trị k lý tưởng nhất phải bằng chính tổng số đường kẻ viền của các ô vuông trên bàn cờ đó.

## Bài 3 – RANSAC với Hình dạng Tuỳ chọn

### 3.1 Chọn Hình dạng và Giải thích

#### Hình dạng 1: Sóng Sine (Sine Wave) — s = 3

**Mô hình:** $y = A \cdot \sin(\omega x + \phi)$ với 3 tham số $(A, \omega, \phi)$

**Xuất hiện tự nhiên:**

Hình dạng này là nền tảng trong xử lý tín hiệu và xuất hiện phổ biến trong các cấu trúc lượn sóng thực tế. Ví dụ điển hình nhất là khi phân tích **Spectrogram** (phổ tần số) của âm thanh hoặc rung động máy móc, nơi các tín hiệu tuần hoàn tạo thành các đường sóng. Trong thị giác máy tính, sóng sine xuất hiện ở các bề mặt vật liệu đặc thù như mái tôn, mặt nước hoặc các nếp gấp vân vải lanh khi quan sát từ một góc nghiêng.

**Tại sao RANSAC phù hợp hơn?**

- **Hough Transform thất bại vì:** Một đường sine cần ít nhất 3 tham số $(A, \omega, \phi)$. Việc dùng Hough đòi hỏi Accumulator **3D**, gây bùng nổ về bộ nhớ và rất dễ sai số do rời rạc hóa (quantization), khiến việc tìm đỉnh không chính xác.

- **Least Squares thất bại vì:** Trong các bài toán thực tế như phân tích phổ, dữ liệu thường bị nhiễu bởi các âm thanh tạp hoặc lỗi đo lường (outliers). Phương pháp này bình phương sai số của các điểm nhiễu, khiến đường sine bị "kéo lệch" hoàn toàn khỏi quỹ đạo tuần hoàn thực tế, làm sai lệch tần số cần tìm.

- **Ưu thế của RANSAC:** Chỉ cần lấy mẫu ngẫu nhiên **3 điểm** để xác định một mô hình ứng viên. Cơ chế "đồng thuận" giúp thuật toán tìm ra đúng quy luật tuần hoàn bền vững của tín hiệu ngay cả khi nhiễu chiếm tới **50–60%** dữ liệu.

#### Hình dạng 2: Hình Elip cân (Axis-aligned Ellipse) — s = 4

**Mô hình:** $\frac{(x-c_x)^2}{a^2} + \frac{(y-c_y)^2}{b^2} = 1$ với 4 tham số $(c_x, c_y, a, b)$

**Xuất hiện tự nhiên:**

Đây là hình dạng cốt lõi trong các hệ thống **Eye-tracking** (theo dõi ánh mắt). Con ngươi vốn có hình tròn, nhưng dưới góc nhìn phối cảnh khi người dùng nhìn nghiêng, nó luôn hiển thị trên ảnh dưới dạng hình elip. Ngoài ra, elip cân còn xuất hiện khi kiểm tra chất lượng các sản phẩm công nghiệp hình tròn (miệng chai, linh kiện) từ các góc máy không trực diện.

**Tại sao RANSAC phù hợp hơn?**

- **Hough Transform thất bại vì:** Elip cân đòi hỏi Accumulator **4 chiều** $(c_x, c_y, a, b)$. Chi phí tính toán và bộ nhớ cho mảng 4D là quá lớn, khiến thuật toán chạy cực chậm và không thể đáp ứng yêu cầu thời gian thực.

- **Least Squares thất bại vì:** Trong bài toán theo dõi mắt, lông mi hoặc các điểm phản chiếu ánh sáng (glints) trên giác mạc đóng vai trò là outliers nằm ngay sát biên elip. Least Squares sẽ cố gắng "chiều lòng" cả những điểm nhiễu này, làm tâm elip bị lệch và khiến việc ước lượng hướng nhìn bị sai sót nghiêm trọng.

- **Ưu thế của RANSAC:** Bằng cách chỉ lấy mẫu tối thiểu **4 điểm** biên, RANSAC nhanh chóng tìm ra tập hợp các điểm thực sự thuộc về con ngươi và bỏ qua hoàn toàn các pixel nhiễu từ lông mi. Điều này đảm bảo tính ổn định và độ chính xác cực cao cho hệ thống trong điều kiện ánh sáng phức tạp.

> **Lưu ý:** Với $s < 5$, số lần lặp $N$ của RANSAC chỉ khoảng **35–70 lần** (với 50% outlier, $p=0.99$), giúp thuật toán đạt hiệu suất thời gian thực so với việc dò tìm trong không gian tham số của Hough Transform.

## 3.4 Số iteration

In [2]:
## Hàm tính số iteration

def compute_iterations(
        outlier_ratio,
        sample_size,
        p=0.99
):

    e = outlier_ratio
    s = sample_size

    numerator = np.log(1 - p)
    denominator = np.log(
        1 - (1 - e)**s
    )

    N = int(
        np.ceil(
            numerator / denominator
        )
    )

    return N

In [3]:
## Kiểm tra bảng trong PDF

outliers = [0.1, 0.3, 0.5, 0.7]

for e in outliers:

    N = compute_iterations(
        outlier_ratio=e,
        sample_size=2
    )

    print(
        f"Outlier={e*100:.0f}% "
        f"-> N={N}"
    )

Outlier=10% -> N=3
Outlier=30% -> N=7
Outlier=50% -> N=17
Outlier=70% -> N=49


## 3.5 Thực nghiệm

In [4]:
## Kiểm tra RANSAC thành công bao nhiêu %

def experiment(
        outlier_ratio,
        sample_size,
        epsilon,
        trials=100
):

    N = compute_iterations(
        outlier_ratio,
        sample_size
    )

    success = 0

    for _ in range(trials):

        n_out = int(
            100 * outlier_ratio
            / (1 - outlier_ratio)
        )

        points = generate_line_data(
            n_inliers=100,
            n_outliers=n_out
        )

        model, inliers = ransac(
            points,
            fit_line,
            line_distance,
            sample_size,
            epsilon,
            N
        )

        if len(inliers) > 80:
            success += 1

    return success / trials

In [5]:
## Chạy với N lý thuyết

for e in [0.2, 0.4, 0.6]:

    rate = experiment(
        e,
        sample_size=2,
        epsilon=0.6,
        trials=100
    )

    print(
        f"Outlier={e*100:.0f}% "
        f"Success={rate*100:.2f}%"
    )

NameError: name 'generate_line_data' is not defined

In [6]:
## Giảm N xuống

def experiment_with_custom_N(
        outlier_ratio,
        sample_size,
        epsilon,
        N,
        trials=100
):

    success = 0

    for _ in range(trials):

        n_out = int(
            100 * outlier_ratio
            / (1 - outlier_ratio)
        )

        points = generate_line_data(
            100,
            n_out
        )

        model, inliers = ransac(
            points,
            fit_line,
            line_distance,
            sample_size,
            epsilon,
            N
        )

        if len(inliers) > 80:
            success += 1

    return success / trials

In [7]:
## So sánh

e = 0.5

N_theory = compute_iterations(
    e,
    sample_size=2
)

for factor in [
        1.0,
        0.75,
        0.5,
        0.25
]:

    N = int(
        N_theory * factor
    )

    rate = experiment_with_custom_N(
        e,
        sample_size=2,
        epsilon=0.6,
        N=N
    )

    print(
        f"N={N} "
        f"Success={rate*100:.2f}%"
    )

NameError: name 'generate_line_data' is not defined

In [8]:
## Vẽ biểu đồ

Ns = []
rates = []

for factor in np.arange(
        0.2,
        1.2,
        0.1
):

    N = int(
        N_theory * factor
    )

    rate = experiment_with_custom_N(
        e,
        sample_size=2,
        epsilon=0.6,
        N=N
    )

    Ns.append(N)
    rates.append(rate)

plt.plot(
    Ns,
    rates,
    marker='o'
)

plt.xlabel(
    "Number of Iterations"
)

plt.ylabel(
    "Success Rate"
)

plt.title(
    "RANSAC Success vs Iterations"
)

plt.grid(True)

plt.show()

NameError: name 'generate_line_data' is not defined